# The pooling problem and the pq-formulation

The **pooling problem** blends raw inputs of known quality through intermediate
*pools* into products with quality specifications, maximizing profit. Because the
blended quality at a pool is a *bilinear* function of the inflows, the problem is a
nonconvex NLP — even tiny instances are hard for global solvers when modeled naively.
The classic benchmark is Haverly's Pooling Problem 1 {cite:p}`Haverly1978`, whose
global optimum is a profit of **400**.

discopt's `discopt.pooling` module builds the **pq-formulation** of
{cite:t}`Tawarmalani2002`. It replaces raw pool inflows with *proportion* variables
$q_{i,\ell}$ (the fraction of pool $\ell$'s inflow from input $i$) and adds the
Reformulation–Linearization (RLT) cuts

$$ \sum_i w_{i,\ell,j} = y_{\ell,j}, \qquad w_{i,\ell,j} = q_{i,\ell}\,y_{\ell,j}, $$

one per pool→output arc. These redundant-but-tightening equalities make the McCormick
{cite:p}`McCormick1976` relaxation of the pq-formulation provably at least as tight as
the textbook *p*-formulation — and on Haverly the root relaxation is already exact, so
the global optimum is certified at the root node.


## Haverly Pooling Problem 1

`haverly_hpp1()` returns the canonical instance: one pool blends two sulfur-bearing
sources, and a third source bypasses the pool to the second product. We build the
pq-formulation and solve it to global optimality.


In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

from discopt.pooling import haverly_hpp1, build_pq_formulation

problem = haverly_hpp1()
model = build_pq_formulation(problem)
result = model.solve()
print(f"status      : {result.status}")
print(f"profit      : {float(result.objective):.2f}   (known optimum: 400)")
print(f"B&B nodes   : {result.node_count}   (root-tight relaxation)")


## Building a custom instance

`PoolingProblem` describes any standard pooling network from `Input`/`Pool`/`Output`
components and arc lists. Here a cheap source is high-sulfur, so the quality spec
forces a 50/50 blend; the optimal profit is 700.


In [ ]:
from discopt.pooling import Input, Pool, Output, PoolingProblem, build_pq_formulation

problem = PoolingProblem(
    inputs=[
        Input("s0", cost=5.0, quality={"sulfur": 1.0}),
        Input("s1", cost=1.0, quality={"sulfur": 3.0}),  # cheap but high-sulfur
    ],
    pools=[Pool("pool")],
    outputs=[Output("p", price=10.0, demand=100.0, quality_max={"sulfur": 2.0})],
    pool_inputs=[("s0", "pool"), ("s1", "pool")],
    pool_outputs=[("pool", "p")],
)
result = build_pq_formulation(problem).solve()
print(f"profit: {float(result.objective):.2f}   (optimal 50/50 blend -> 700)")


## Why the pq cuts matter

The proportion variables and the RLT equalities $\sum_i w_{i,\ell,j} = y_{\ell,j}$ are
what separate the pq-formulation from the weak *p*-formulation: they couple the pool
throughput to the blend proportions, tightening every bilinear McCormick envelope
simultaneously. Reusing discopt's existing spatial branch-and-bound unchanged, the
formulation only *tightens* the relaxation — it never alters the feasible region or the
optimum {cite:p}`Tawarmalani2002`.
